# TurboLLM on Kaggle — dual T4 (auto-tune vs chat)

Reproduce & verify the two consumer complaints on a real 2×T4 box: (a) both GPUs are
recognised & used everywhere, and (b) chat tok/s matches the tok/s auto-tune reports.

**Before you run anything:** top-right **Settings** →
- **Accelerator = GPU T4 x2**
- **Internet = On**

Then run the sections **top to bottom**. Section 2 builds a CUDA engine the *first* time
(~30–40 min) and caches it in `/kaggle/working`; every later run skips straight past it
(~1–2 min). Each section is idempotent — safe to re-run.

## 0 · Preflight — fail fast if the box isn't set up
Checks that 2 GPUs are attached and the internet is on, so you never waste build time on a
misconfigured session.

In [ ]:
import subprocess, urllib.request
smi = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout.strip() or smi.stderr.strip())
n = len([l for l in smi.stdout.strip().splitlines() if l.strip()])
assert n >= 2, f"Need 2 GPUs (Settings -> Accelerator -> GPU T4 x2); found {n}"
try:
    urllib.request.urlopen("https://huggingface.co", timeout=8)
except Exception as e:
    raise SystemExit(f"Internet looks OFF (Settings -> Internet -> On): {e}")
print(f"\nPreflight OK — {n} GPUs, internet on.")

## 1 · Get the code (branch)
Clones the first time, fast-forwards after. Re-run this to pull the latest fixes.

In [ ]:
!git clone -b claude/turbollm-runpod-dual-gpu-7aa8c0 https://github.com/mohitsoni48/TurboLLM.git 2>/dev/null; cd /kaggle/working/TurboLLM && git pull && git log --oneline -1

## 2 · Setup (one-time CUDA build + model, cached)
Node, npm deps, web UI, a native **CUDA** build of the TurboQuant llama.cpp fork, and the
Q4 model. **First run ~30–40 min**; later runs print `✓ already built / present` and finish
in ~1–2 min. If a session dies mid-build, just re-run — it resumes.

In [ ]:
!bash /kaggle/working/TurboLLM/deploy/kaggle/setup.sh

## 3 · Serve — daemon + CUDA engine + public GUI tunnel
Starts the daemon from source, registers & activates the CUDA engine, and opens a public
`*.trycloudflare.com` tunnel. **Open the printed URL and enter the printed Token** to use
the real TurboLLM GUI (Section 6 lists what to verify there).

In [ ]:
!bash /kaggle/working/TurboLLM/deploy/kaggle/serve.sh start

## 4 · The test — auto-tune vs chat tok/s + dual-GPU
Runs the built-in auto-tune, saves the winning profile, loads it, measures **real streaming
chat tok/s**, samples **both** T4s during generation, and diffs winner-vs-loaded config.
Prints auto-tune tps vs chat tps side by side. Takes a few minutes (the auto-tune sweep).

In [ ]:
!cd /kaggle/working/TurboLLM && python3 deploy/kaggle/bench_vs_chat.py --ctx 8192

## 5 · Dual-GPU verification (API)
`sysinfo` must list 2 GPUs; the active engine must be **TurboQuant CUDA (T4)**. Run the
`nvidia-smi` line *while Section 4 is generating* to see non-zero memory + util on **both**
GPU 0 and GPU 1.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!echo; echo '--- GPUs seen by TurboLLM (expect 2) ---'; curl -s localhost:6996/api/v1/sysinfo | python3 -c "import json,sys; d=json.load(sys.stdin); [print(g.get('name'), round(g.get('vramMb',0)/1024), 'GB', g.get('vendor')) for g in d.get('gpus',[])]"
!echo '--- active engine ---'; curl -s localhost:6996/api/v1/engines | python3 -c "import json,sys; d=json.load(sys.stdin); a=d.get('activeEngineId'); [print(('* ' if e['id']==a else '  ')+e['name']) for e in d.get('engines',[])]"

## 6 · GUI checklist (open the tunnel URL from Section 3)

Verify dual-GPU is shown/used at **every** surface:

1. **Engines screen** → hardware line reads **`2× Tesla T4 · 30 GB · Linux`**, active engine = **TurboQuant CUDA (T4)**.
2. **Model → config** → GPU split / offload controls present and editable.
3. **Auto-tune** → winning tok/s shown (compare to Section 4).
4. **Chat** → send a message; re-run Section 5's `nvidia-smi` line and confirm **both** GPUs are busy.

### Dev loop
- src change: re-run Section 1, then `!cd /kaggle/working/TurboLLM && bash deploy/kaggle/serve.sh restart`
- web change: re-run Section 1, then re-run Section 2 (it rebuilds the UI when the commit changed) and `serve.sh restart`.

Knobs: `TURBOLLM_MODEL_FILE` (default `Qwen3.6-27B-Q4_K_M.gguf`), `--model KEY` / `--ctx N` on the test script.